# Phase 1 Measurement Workflow (Debug/Bring-up)

This notebook is intentionally short and operational:
- one setup block
- one preflight-only block
- one run+process+save block
- one plot block for last run
- one plot block for last N runs

Use this notebook for early-stage debugging and routine measurement execution.

In [ ]:
# BLOCK 1 - Setup (run once)
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys

import numpy as np


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "eis").exists() and (candidate / "USB6451").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from current working directory.")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from eis import (
    CaptureConditioningConfig,
    ExcitationConfig,
    HardwareConfig,
    ImpedanceProcessingConfig,
    RunSelection,
    USB6451Adapter,
    build_artifact_link_payload,
    build_metadata_bank,
    compute_impedance_for_run,
    create_run_folder_layout,
    execute_sweep,
    load_and_validate_config,
    persist_run_artifacts,
    plot_impedance_bode,
    plot_impedance_inverse_nyquist,
    plot_impedance_nyquist,
    plot_raw_vs_fitted_from_csv,
    plot_snr_vs_frequency,
    run_preflight_check,
    write_description_file,
    write_metadata_bank_csv,
    write_metadata_bank_txt,
    write_metadata_report_html,
    write_metadata_report_pdf,
)
from eis.models.measurement_models import PreflightCheckResult

# ----------------------------- User Inputs -----------------------------
CONFIG_PATH = REPO_ROOT / "config_examples" / "config_phase1_example.xlsx"
BASE_OUTPUT_DIR = REPO_ROOT / "measurements"
SERIAL_NUMBER = f"PH1_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
USER_NAME = "operator"
DESCRIPTION = ""
REPEATS = 1
RUN_WITH_FAKE_ADAPTER = True
RUN_PREFLIGHT_DURING_SWEEP = False

# ----------------------------- Save Options ----------------------------
SAVE_METADATA_BANK_TXT = True
SAVE_METADATA_BANK_CSV = True
SAVE_METADATA_REPORT_HTML = True
SAVE_METADATA_REPORT_PDF = False
SAVE_DESCRIPTION_FILE = True
SAVE_PLOTS_PNG = True
SAVE_PLOTS_VECTOR = True

# ------------------------ Hardware/Excitation --------------------------
hardware = HardwareConfig(
    device="Dev1",
    ao_channel="ao0",
    ai_channels=("ai0", "ai7"),
    input_mode="differential",
)

excitation = ExcitationConfig(
    drive_mode="auto_from_current_rms",
    offset_v=0.0,
    manual_current_range="20A",  # or None for auto
)

conditioning = CaptureConditioningConfig(
    settle_discard_s=0.15,
    extra_periods_for_trim=1,
    alignment_search_periods=1,
)

processing = ImpedanceProcessingConfig(
    method="fft",                     # "fft" or "sine_fit"
    sine_fit_backend="numpy_lstsq",  # "numpy_lstsq" or "scipy_least_squares"
    filter_mode="lowpass",           # "none", "lowpass", "bandpass"
    lowpass_cutoff_hz=2000.0,
    shunt_resistance_ohm=0.008,
)

# ----------------------------- Preflight -------------------------------
PREFLIGHT_SAMPLE_RATE_SPS = None  # None => use first row sample rate
PREFLIGHT_SAMPLES_PER_CHANNEL = None  # None => auto-sized from discard window
PREFLIGHT_TEST_CURRENT_RMS_A = 10.0
PREFLIGHT_MANUAL_CURRENT_RANGE = "20A"  # None => fall back to excitation range/auto
PREFLIGHT_SHUNT_RESISTANCE_OHM = 0.008
PREFLIGHT_SHUNT_TOLERANCE_PERCENT = 15.0
PREFLIGHT_CURRENT_CHANNEL_INDEX = 0
PREFLIGHT_SETTLE_DISCARD_S = 0.15

# ------------------------- Plot Selection ------------------------------
LAST_N_FOR_OVERLAY = 3
SERIAL_FILTER_FOR_OVERLAY = SERIAL_NUMBER
RAW_PLOT_ROW_NUMBER = 2
RAW_PLOT_REPEAT_INDEX = 1


class FakeAdapter:
    """Hardware-free adapter for notebook debugging without NI DAQ."""

    def run_preflight_check(self, **kwargs) -> PreflightCheckResult:
        samples = int(kwargs["samples_per_channel"])
        return PreflightCheckResult(
            sample_rate_sps=float(kwargs["sample_rate_sps"]),
            samples_per_channel=samples,
            measured_shape=(2, samples),
            message="Fake preflight passed",
        )

    def measure_sine_point(self, **kwargs):
        frequency_hz = float(kwargs["frequency_hz"])
        sample_rate_sps = float(kwargs["sample_rate_sps"])
        sample_count = int(round(float(kwargs["n_periods"]) * sample_rate_sps / frequency_hz))
        sample_count = max(64, sample_count)

        t = np.arange(sample_count, dtype=np.float64) / sample_rate_sps
        omega = 2.0 * np.pi * frequency_hz

        i_peak_a = 1.6
        i_phase_rad = 0.15
        r_shunt_ohm = 0.008
        v_shunt = (r_shunt_ohm * i_peak_a) * np.sin(omega * t + i_phase_rad)

        z_dut = 5.0 + 1.2j
        v_dut_peak = i_peak_a * abs(z_dut)
        v_dut_phase = i_phase_rad + float(np.angle(z_dut))
        v_dut = v_dut_peak * np.sin(omega * t + v_dut_phase)

        rng = np.random.default_rng(int(20260303 + round(frequency_hz * 10.0)))
        v_shunt = v_shunt + rng.normal(loc=0.0, scale=2.5e-4, size=sample_count)
        v_dut = v_dut + rng.normal(loc=0.0, scale=8.0e-3, size=sample_count)
        return np.vstack([v_shunt, v_dut])

    def close(self) -> None:
        return None


adapter = FakeAdapter() if RUN_WITH_FAKE_ADAPTER else USB6451Adapter()
sweep = load_and_validate_config(CONFIG_PATH)

print(f"Repo root: {REPO_ROOT}")
print(f"Config rows: {len(sweep.points)}")
print(f"Run mode: {'FAKE' if RUN_WITH_FAKE_ADAPTER else 'REAL HARDWARE'}")
print(f"Serial number: {SERIAL_NUMBER}")

In [ ]:
# BLOCK 2 - Preflight Only (no measurement sweep)
preflight_rate = (
    float(PREFLIGHT_SAMPLE_RATE_SPS)
    if PREFLIGHT_SAMPLE_RATE_SPS is not None
    else float(sweep.points[0].sample_rate_sps)
)

if PREFLIGHT_SAMPLES_PER_CHANNEL is None:
    required_settle_samples = int(round(PREFLIGHT_SETTLE_DISCARD_S * preflight_rate))
    analysis_samples = max(64, int(round(0.02 * preflight_rate)))
    preflight_samples = required_settle_samples + analysis_samples
else:
    preflight_samples = int(PREFLIGHT_SAMPLES_PER_CHANNEL)

preflight_only_result = run_preflight_check(
    adapter=adapter,
    hardware=hardware,
    sample_rate_sps=preflight_rate,
    samples_per_channel=preflight_samples,
    test_current_rms_a=PREFLIGHT_TEST_CURRENT_RMS_A,
    manual_current_range=PREFLIGHT_MANUAL_CURRENT_RANGE,
    range_selection_policy=excitation.range_selection_policy,
    shunt_resistance_ohm=PREFLIGHT_SHUNT_RESISTANCE_OHM,
    shunt_voltage_tolerance_percent=PREFLIGHT_SHUNT_TOLERANCE_PERCENT,
    current_channel_index=PREFLIGHT_CURRENT_CHANNEL_INDEX,
    settle_discard_s=PREFLIGHT_SETTLE_DISCARD_S,
)

print("Preflight-only completed")
print(f"  Sample rate: {preflight_only_result.sample_rate_sps:g} S/s")
print(f"  Samples/ch : {preflight_only_result.samples_per_channel}")
print(f"  Shape      : {preflight_only_result.measured_shape}")
print(f"  Message    : {preflight_only_result.message}")

In [ ]:
# BLOCK 3 - Run, process, save (+ validation summary)
run_result = execute_sweep(
    sweep=sweep,
    adapter=adapter,
    hardware=hardware,
    excitation=excitation,
    repeats=REPEATS,
    run_preflight=RUN_PREFLIGHT_DURING_SWEEP,
    preflight_sample_rate_sps=PREFLIGHT_SAMPLE_RATE_SPS,
    preflight_samples_per_channel=PREFLIGHT_SAMPLES_PER_CHANNEL,
    preflight_test_current_rms_a=PREFLIGHT_TEST_CURRENT_RMS_A,
    preflight_manual_current_range=PREFLIGHT_MANUAL_CURRENT_RANGE,
    preflight_shunt_resistance_ohm=PREFLIGHT_SHUNT_RESISTANCE_OHM,
    preflight_shunt_voltage_tolerance_percent=PREFLIGHT_SHUNT_TOLERANCE_PERCENT,
    preflight_current_channel_index=PREFLIGHT_CURRENT_CHANNEL_INDEX,
    preflight_settle_discard_s=PREFLIGHT_SETTLE_DISCARD_S,
    conditioning=conditioning,
)

impedance_results = compute_impedance_for_run(run_result=run_result, config=processing)
layout = create_run_folder_layout(
    base_output_dir=BASE_OUTPUT_DIR,
    serial_number=SERIAL_NUMBER,
    started_at_local=datetime.now(),
)

persisted = persist_run_artifacts(
    layout=layout,
    run_result=run_result,
    impedance_results=impedance_results,
)
capture_artifacts, point_summaries = build_artifact_link_payload(persisted)

metadata_bank = build_metadata_bank(
    sweep=sweep,
    run_result=run_result,
    hardware=hardware,
    excitation=excitation,
    serial_number=SERIAL_NUMBER,
    user_name=USER_NAME,
    description=DESCRIPTION,
    capture_artifacts=capture_artifacts,
    point_summaries=point_summaries,
)

saved_paths = []
if SAVE_METADATA_BANK_TXT:
    saved_paths.append(write_metadata_bank_txt(metadata_bank, layout.root / "metadata_bank.txt"))
if SAVE_METADATA_BANK_CSV:
    saved_paths.append(write_metadata_bank_csv(metadata_bank, layout.root / "metadata_measurements.csv"))
if SAVE_METADATA_REPORT_HTML:
    saved_paths.append(write_metadata_report_html(metadata_bank, layout.reports / "metadata_report.html"))
if SAVE_METADATA_REPORT_PDF:
    saved_paths.append(write_metadata_report_pdf(metadata_bank, layout.reports / "metadata_report.pdf"))
if SAVE_DESCRIPTION_FILE:
    path = write_description_file(DESCRIPTION, layout.root / "description.txt")
    if path is not None:
        saved_paths.append(path)

LAST_RUN_ROOT = layout.root
LAST_RUN_PLOTS_DIR = layout.plots
LAST_RUN_SERIAL = SERIAL_NUMBER

capture_freq_map = {
    (c.row_number, c.repeat_index): float(c.frequency_hz)
    for c in run_result.captures
}

adapter.close()

print("Run completed")
print(f"  Run folder          : {LAST_RUN_ROOT}")
print(f"  Captures            : {len(run_result.captures)}")
print(f"  Impedance rows      : {len(impedance_results)}")
print(f"  Raw artifacts       : {len(persisted.capture_artifacts)}")
print(f"  Point summaries     : {len(persisted.point_summaries)}")
if run_result.preflight is None:
    print("  Sweep preflight     : skipped (preflight-only block can be used separately)")
else:
    print(f"  Sweep preflight     : {run_result.preflight.message}")
print("  Saved files:")
for item in saved_paths:
    print(f"    - {item}")

In [ ]:
# BLOCK 4 - Plots for last run
if "LAST_RUN_ROOT" not in globals():
    raise RuntimeError("Run BLOCK 3 first so LAST_RUN_ROOT is available.")

selection_last = RunSelection(mode="last", serial_numbers=(LAST_RUN_SERIAL,))

saved_plot_paths = []

def _png(name: str):
    if not SAVE_PLOTS_PNG:
        return None
    return LAST_RUN_PLOTS_DIR / f"{name}.png"


nyquist_png = _png("last_run_nyquist")
inv_nyquist_png = _png("last_run_inverse_nyquist")
bode_png = _png("last_run_bode")
snr_png = _png("last_run_snr")

plot_impedance_nyquist(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last,
    save_path=nyquist_png,
)
plot_impedance_inverse_nyquist(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last,
    save_path=inv_nyquist_png,
)
plot_impedance_bode(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last,
    save_path=bode_png,
)
plot_snr_vs_frequency(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last,
    snr_source="current",
    threshold_db=20.0,
    good_region="above_threshold",
    save_path=snr_png,
)

for item in (nyquist_png, inv_nyquist_png, bode_png, snr_png):
    if item is not None:
        saved_plot_paths.append(item)

# Raw-vs-fitted for one selected row/repeat from this run
raw_item = None
for artifact in persisted.capture_artifacts:
    if artifact.row_number == RAW_PLOT_ROW_NUMBER and artifact.repeat_index == RAW_PLOT_REPEAT_INDEX:
        raw_item = artifact
        break

if raw_item is None:
    raise ValueError(
        f"No raw artifact found for row={RAW_PLOT_ROW_NUMBER}, repeat={RAW_PLOT_REPEAT_INDEX}."
    )

raw_csv_path = LAST_RUN_ROOT / raw_item.raw_csv_relpath
raw_frequency_hz = capture_freq_map[(raw_item.row_number, raw_item.repeat_index)]

raw_fit_png = _png(f"last_run_raw_vs_fit_row{RAW_PLOT_ROW_NUMBER}_rep{RAW_PLOT_REPEAT_INDEX}")
raw_fit_svg = (
    LAST_RUN_PLOTS_DIR / f"last_run_raw_vs_fit_row{RAW_PLOT_ROW_NUMBER}_rep{RAW_PLOT_REPEAT_INDEX}.svg"
    if SAVE_PLOTS_VECTOR
    else None
)

plot_raw_vs_fitted_from_csv(
    raw_csv_path=raw_csv_path,
    frequency_hz=raw_frequency_hz,
    save_path=raw_fit_png,
    save_vector_path=raw_fit_svg,
    title=f"Raw vs Fitted | row={RAW_PLOT_ROW_NUMBER}, repeat={RAW_PLOT_REPEAT_INDEX}",
)

if raw_fit_png is not None:
    saved_plot_paths.append(raw_fit_png)
if raw_fit_svg is not None:
    saved_plot_paths.append(raw_fit_svg)

print("Last-run plots generated")
for item in saved_plot_paths:
    print(f"  - {item}")

In [ ]:
# BLOCK 5 - Plots for last N runs (set N here)
if "LAST_RUN_ROOT" not in globals():
    raise RuntimeError("Run BLOCK 3 first so output paths are available.")

N = int(LAST_N_FOR_OVERLAY)
if N < 1:
    raise ValueError("LAST_N_FOR_OVERLAY must be >= 1")

selection_last_n = RunSelection(
    mode="last_n",
    last_n=N,
    serial_contains=SERIAL_FILTER_FOR_OVERLAY,
)

nyquist_last_n = LAST_RUN_PLOTS_DIR / f"last_{N}_nyquist.png"
bode_last_n = LAST_RUN_PLOTS_DIR / f"last_{N}_bode.png"
snr_last_n = LAST_RUN_PLOTS_DIR / f"last_{N}_snr.png"

_, _, runs_ny = plot_impedance_nyquist(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last_n,
    save_path=(nyquist_last_n if SAVE_PLOTS_PNG else None),
)
_, _, runs_bd = plot_impedance_bode(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last_n,
    save_path=(bode_last_n if SAVE_PLOTS_PNG else None),
)
_, _, runs_sr, _ = plot_snr_vs_frequency(
    base_output_dir=BASE_OUTPUT_DIR,
    selection=selection_last_n,
    snr_source="current",
    threshold_db=20.0,
    good_region="above_threshold",
    save_path=(snr_last_n if SAVE_PLOTS_PNG else None),
)

print(f"Last-{N} overlay plots generated")
print("  Runs used (Nyquist):")
for item in runs_ny:
    print(f"    - {item.root.name}")
print("  Runs used (Bode):")
for item in runs_bd:
    print(f"    - {item.root.name}")
print("  Runs used (SNR):")
for item in runs_sr:
    print(f"    - {item.root.name}")
if SAVE_PLOTS_PNG:
    print(f"  - {nyquist_last_n}")
    print(f"  - {bode_last_n}")
    print(f"  - {snr_last_n}")